# 📘 부록: OKF(Open Knowledge Format)와 LLM 위키

OKF(Open Knowledge Format)는 지식을 마크다운 파일의 폴더로 나타내는 공개 형식입니다. 사람과 AI 에이전트가 모두 읽고 쓸 수 있게 만들었고, Google Cloud가 공개 사양으로 발표했습니다(이 부록은 v0.2 기준). 이 부록은 OKF가 무엇을 위해 있고 어떻게 쓰는지부터 공식 문서 기준으로 보고, 예시 번들 두 개(업무 위키, 데이터 위키)로 구조와 규칙을 봅니다.

**공식 문서:** [OKF SPEC v0.2](https://github.com/GoogleCloudPlatform/open-knowledge-format/blob/main/SPEC.md), [Google Cloud 블로그](https://cloud.google.com/blog/products/data-analytics/how-the-open-knowledge-format-can-improve-data-sharing/), [Karpathy LLM Wiki](https://gist.github.com/karpathy/442a6bf555914893e9891c11519de94f)

**학습 목표**

1. **OKF가 무엇을 위해 있고 무엇에 쓰는지 설명합니다.**
2. **LLM 위키의 세 층과 세 동작, OKF의 설계 원칙 세 가지를 설명합니다.**
3. **번들 구조와 개념 문서에 넣는 것, 넣지 않는 것을 구분합니다.**
4. **출처, 검증 기록, 상태, 만료 시점으로 문서를 얼마나 믿을지 판정하는 기준과, 번들이 지켜야 할 적합성 조건을 설명합니다.**

## 1. OKF가 하는 일

### 1-1. LLM 위키가 필요한 상황

가상 게임 회사 별빛픽셀에서 누군가 "정식 출시일은?" 하고 묻는 상황입니다. 출시일에 관한 자료는 세 군데에 있습니다.

- **10월 27일 회의록:** 정식 출시를 11월 20일로 확정했습니다.
- **출시 체크리스트:** 그 회의 뒤에 쓴 문서라 11월 20일이 적혀 있습니다.
- **10월 31일 긴급 메모:** 결제 오류 때문에 정식 출시를 12월 4일로 미뤘습니다.

<img src="images/00_why_llm_wiki.png" width="1050" alt="같은 질문 '정식 출시일은?'에 대한 두 경우. 왼쪽은 회의록(11월 20일), 긴급 메모(12월 4일), 체크리스트(11월 20일)가 흩어져 있어 옛 값 11월 20일을 답합니다. 오른쪽은 긴급 메모가 오자 프로젝트 문서를 12월 4일로 고쳐 두어 12월 4일을 답합니다.">

- **자료만 쌓여 있을 때:** 에이전트는 질문이 올 때마다 자료를 다시 찾아 읽습니다. 날짜가 다른 자료가 함께 나오면 어느 쪽이 최신인지 그때마다 판단해야 하고, 두 자료가 11월 20일이라 옛 값을 고르기 쉽습니다. 같은 질문이 백 번 오면 같은 판단을 백 번 합니다.
- **위키로 정리해 둘 때:** 긴급 메모가 들어온 순간 LLM이 프로젝트 문서 하나를 12월 4일로 고치고, 근거로 긴급 메모를 적어 둡니다. 질문이 오면 그 문서만 읽고 답합니다.

이렇게 원본 자료를 LLM이 미리 문서로 정리해 두고 새 자료가 올 때마다 고치며, 질문에는 그 문서로 답하는 방식이 **LLM 위키**입니다(2절). OKF는 이런 위키를 **한 번 정리하면 어떤 에이전트나 도구도 같은 방식으로 읽는** 공통 형식입니다.

### 1-2. 할 수 있는 일

| 쓰임새 | 번들에 담는 것 | 읽는 쪽 |
|---|---|---|
| 업무 비서 | 사람, 프로젝트, 회의 결정, 업무 절차 | 팀원 질문에 답하고, 새 회의록이나 메일이 오면 그 내용으로 위키 문서(일정, 담당자 등)를 고치는 사내 에이전트 |
| 데이터 카탈로그 | 테이블과 열 설명, 지표 정의 | SQL을 쓰거나 대시보드를 만드는 데이터 에이전트 |
| 운영 절차서(런북) | 장애 대응 순서, 확인과 복구 명령어 | 장애 때 순서를 안내하는 운영 에이전트 |
| 개인 지식 비서 | 여행, 장소, 구독, 메모 | 개인 비서 에이전트 |

## 2. LLM 위키

OKF는 LLM 위키라는 방식에서 나왔습니다. Google Cloud 블로그는 OKF를 'LLM 위키 패턴을 옮겨 쓸 수 있고 서로 읽을 수 있는 형식으로 정식화한 공개 사양'이라고 소개합니다.

**frontmatter:** 마크다운 파일 맨 위에 붙이는 정보 칸입니다. `---` 두 줄 사이에 `키: 값` 꼴(YAML 형식)로 문서 종류, 제목, 상태 같은 문서에 대한 정보를 적고, 그 아래에 사람이 읽는 본문을 씁니다. 본문은 문장이라 해석해야 하지만 frontmatter는 정해진 키와 값이라 코드가 바로 읽습니다. frontmatter는 따로 저장된 칸이 아니라 같은 파일의 맨 위 몇 줄이라, 파일을 열면 본문과 함께 보이고 코드는 두 `---` 사이만 잘라 읽습니다.

~~~markdown
---
type: Project
title: 초롱 원정대 출시
status: stable
---
# 개요

초롱이가 등불을 밝히며 밤하늘 섬을 탐험하는 모바일 퍼즐 어드벤처입니다.
~~~

- **문제:** LLM 위키는 Karpathy의 개인 위키, 팀 위키, 데이터 카탈로그를 내보낸 파일처럼 여러 곳에서 쓰입니다. 모두 마크다운과 frontmatter로 비슷해 보이지만 어떤 키를 넣고 어떤 파일 이름이 무슨 뜻인지가 제각각이라, 한쪽에서 만든 위키를 다른 에이전트가 그대로 읽지 못했습니다.
- **OKF가 정한 것:** 문서마다 있어야 하는 `type` 키, 목차 `index.md`와 변경 기록 `log.md` 같은 예약 파일 이름처럼 꼭 필요한 공통 규칙만 정했습니다.
- **효과:** 이 규칙을 지키면 사람이 쓴 위키든, 에이전트가 쓴 위키든, 내보내기 프로그램이 만든 위키든 다른 에이전트나 도구가 변환 없이 읽습니다.

그래서 OKF를 보기 전에 LLM 위키가 무엇인지부터 봅니다.

### 2-1. RAG와 LLM 위키

RAG는 질문이 올 때마다 원문 조각을 검색해 답에 붙입니다. LLM 위키는 원문을 한 번 읽어 주제별 문서로 정리해 두고, 새 자료가 오면 그 문서들을 고칩니다. 그래서 정리한 지식이 쌓이고, 문서 사이의 링크와 모순 점검도 미리 해 둘 수 있습니다.

<img src="images/01_rag_vs_llm_wiki.png" width="1050" alt="RAG는 질문마다 원문을 검색하고, LLM 위키는 원본 자료를 한 번 정리해 쌓아 둔 위키에서 답합니다.">

**목차로 찾기와 검색으로 찾기**

답에 쓸 문서는 목차에서 고르는 것이 기본이고, 위키가 커지면 아래 순서로 방법을 더합니다.

<img src="images/11_wiki_find_steps.png" width="1050" alt="위키가 커질 때 문서를 찾는 네 단계. 1 목차 읽기(문서 수백 개까지), 2 목차 나누기(맨 위 목차, 폴더 목차, 문서 순서), 3 목차 줄 검색(제목과 한 줄 설명), 4 본문 검색(세부 내용까지). 어느 단계든 정리된 위키를 찾습니다.">

1. **목차 읽기(기본):** `index.md`의 한 줄 설명을 보고 필요한 문서만 엽니다(2-4). Karpathy는 위키 문서 수백 개까지는 이것으로 충분하다고 봅니다.
2. **목차 나누기:** 목차를 두 단계로 나눠, 폴더마다 `index.md`를 하나씩 둡니다.
    - **맨 위 목차:** 위키 폴더 바로 아래에 있는 `index.md`입니다. 문서는 적지 않고 하위 폴더만 한 줄씩 적습니다. 예: 업무 위키의 `studio/index.md`에는 사람, 프로젝트, 회의, 업무 절차, 결정의 다섯 폴더가 적혀 있습니다.
    - **폴더 목차:** 하위 폴더마다 그 안에 있는 `index.md`입니다. 그 폴더의 문서만 한 줄씩 적습니다. 예: `studio/playbooks/index.md`에는 출시 체크리스트, 서버 장애 대응, 휴가 신청의 세 문서가 적혀 있습니다.
    - **찾는 순서:** 서버 장애를 물으면 에이전트는 맨 위 목차(5줄)에서 '업무 절차' 폴더를 고르고, 그 폴더 목차(3줄)에서 '서버 장애 대응'을 골라 엽니다. 문서 15개가 모두 적힌 긴 목차 하나 대신 짧은 목차 두 개만 읽습니다. 이 부록의 번들이 이 방식입니다.
3. **목차 줄 검색:** 목차도 길어지면 목차 줄(제목과 한 줄 설명)을 검색해 후보 문서를 고릅니다.
4. **본문 검색:** 답이 한 줄 설명에 없는 세부 내용에만 있으면 본문까지 검색합니다. 예를 들어 "사내 테스트 계정은 빼?"의 답은 지표 목차에는 없고 DAU와 7일 잔존율 문서 본문에 있습니다. Karpathy가 소개한 [qmd](https://github.com/tobi/qmd)가 이런 도구입니다(키워드 검색, 벡터 검색, LLM 리랭킹).

어느 단계든 검색 대상은 정리된 위키이고 원본 자료가 아닙니다. 이 점이 RAG와 다릅니다. 검색 도구는 에이전트가 명령줄(CLI)이나 MCP 도구로 불러 후보를 받고, 문서는 에이전트가 직접 엽니다.

이 부록의 위키는 문서가 수십 개라 목차만 씁니다.

### 2-2. 세 층과 세 동작

- **원본 자료(raw sources):** 회의록, 메일, 표 같은 원래 자료입니다. LLM은 읽기만 하고 고치지 않습니다.
- **위키(wiki), OKF에서는 번들(bundle):** LLM이 쓰고 고치는 마크다운 문서 묶음입니다. 사람은 주로 읽습니다. OKF는 이 위키 하나를 번들이라고 부르며, 이 부록에서 번들이라고 하면 곧 위키입니다(4-1).
- **스키마(schema):** LLM에게 위키를 어떻게 쓰라고 알려 주는 규칙 문서입니다(어떤 폴더에 어떤 문서를 두는지, 문서 양식, 넣기와 점검 순서). Karpathy는 `CLAUDE.md`, `AGENTS.md` 같은 에이전트 지침 파일을 예로 듭니다.

예를 들어 팀원이 "신제품 출시일이 언제야?"라고 묻는다고 합시다.

- **원본 자료:** 3월 2일 회의록에는 "출시일은 4월 10일로 확정", 3월 20일 메일에는 "출시를 4월 24일로 미룹니다"가 적혀 있습니다. 답하려면 관련 기록을 모두 찾아 어느 쪽이 최신인지 가려야 합니다.
- **위키:** `신제품 출시` 문서(`projects/launch.md`) 한 곳에 "출시일: 4월 24일"만 적혀 있고, 각주가 근거인 3월 20일 메일을 가리킵니다. 새 자료가 오면 LLM이 이 값을 고칩니다.

원본 자료는 날짜순으로 쌓인 기록이고, 위키는 대상별로 지금 맞는 내용만 정리한 문서입니다. 원본을 검색하는 RAG는 옛 회의록 조각을 가져와 4월 10일로 답할 수 있지만, 위키를 읽으면 최신 값 하나만 보입니다.

<img src="images/02_llm_wiki_layers.png" width="1050" alt="LLM 위키는 원본 자료, 위키, 스키마의 세 층으로 이루어지고 넣기, 묻기, 점검 세 동작으로 위키를 다룹니다.">

- **넣기(ingest):** 새 자료를 읽고 관련 문서를 고치거나 새로 만듭니다. 목차와 변경 기록도 함께 고칩니다.
- **묻기(query):** 목차부터 읽어 필요한 문서만 열고, 출처를 붙여 답합니다.
- **점검(lint):** 모순, 만료된 내용, 끊긴 링크, 들어오는 링크가 없는 문서를 찾습니다.

### 2-3. 새 자료가 오면: 넣기

<img src="images/09_wiki_ingest_flow.png" width="1050" alt="새 자료가 오면 목차에서 관련 문서를 찾아 고치고, 목차와 변경 기록을 고친 뒤 다른 문서와 모순이 없는지 점검합니다. 원본 자료는 그대로 둡니다.">

2-2의 예에서 3월 20일 메일 "출시를 4월 24일로 미룹니다"가 들어오면 LLM은 이렇게 처리합니다.

1. **새 자료 읽기:** 메일을 읽습니다. 원본 자료는 고치지 않고 그대로 둡니다.
2. **관련 문서 찾기:** 목차(`index.md`)의 문서 한 줄 설명을 보고 이 메일로 내용이 바뀌는 문서를 고릅니다. 여기서는 `신제품 출시` 문서입니다. 맞는 문서가 없으면 새로 만듭니다.
3. **문서 고치기:** 출시일을 4월 24일로 고치고, 근거인 메일을 각주로 답니다. 옛 값은 본문에 남기지 않습니다.
4. **목차와 기록 고치기:** 문서 설명이 바뀌었으면 `index.md`를 고치고, `log.md`에 오늘 날짜로 "메일을 반영해 신제품 출시 문서를 고쳤다"를 한 줄 남깁니다. 언제 어떤 자료로 고쳤는지는 이 기록에, 옛 값 자체는 위키를 담은 git 저장소의 이력에 남습니다.
5. **점검:** 고친 문서와 링크로 이어진 문서를 견줍니다. 출시 체크리스트에 아직 4월 10일이 적혀 있으면 모순으로 알리고, 어떻게 고칠지는 사람이 정합니다.

### 2-4. 질문이 오면: 묻기

<img src="images/10_wiki_query_flow.png" width="1050" alt="질문이 오면 맨 위 목차, 폴더 목차를 차례로 읽어 필요한 문서만 열고, 문서 내용으로 답하며 출처를 붙입니다.">

팀원이 "출시일이 언제야?"라고 물으면 LLM은 이렇게 답합니다.

1. **맨 위 목차:** 위키(번들) 맨 위 `index.md`에서 질문에 맞는 폴더(프로젝트)를 고릅니다.
2. **폴더 목차:** 그 폴더의 `index.md`에서 문서마다 적힌 한 줄 설명을 보고 열 문서를 고릅니다.
3. **필요한 문서만:** 고른 문서(`projects/launch.md`)만 엽니다. 원본 자료(회의록, 메일)는 열지 않습니다.
4. **답:** 문서에 적힌 값(4월 24일)으로 답하고, 읽은 문서를 출처로 붙입니다. 문서에 없는 내용은 모른다고 답합니다.

RAG처럼 질문마다 원문 조각을 검색하지 않고, 목차를 따라가 이미 정리된 최신 값만 읽습니다.

## 3. OKF의 설계 원칙

OKF 번들은 **frontmatter가 붙은 마크다운 파일을 폴더에 모아 둔 것**일 뿐입니다. 미리 정해 둔 문서 종류 목록도, 꼭 써야 하는 프로그램도 없습니다. 텍스트 파일이라 메모장으로 열어도 읽을 수 있습니다.

- **규칙은 최소로:** 문서마다 반드시 있어야 하는 키는 `type` 하나입니다. 나머지는 권장이거나 선택입니다.
- **만드는 쪽과 읽는 쪽은 독립:** 사람, 에이전트, 내보내기 파이프라인이 만든 번들을 서로 모르는 도구가 그대로 읽습니다. 그래서 읽는 쪽은 모르는 `type`, 모르는 키, 끊긴 링크가 있어도 번들을 거부하면 안 됩니다.
- **플랫폼이 아니라 형식:** 특정 클라우드, 데이터베이스, 모델, 에이전트 프레임워크에 묶이지 않습니다. git 저장소나 압축 파일로 주고받습니다.

## 4. 번들 구조

### 4-1. 번들과 개념 문서

**번들(bundle)** 은 OKF 형식으로 쓴 위키 하나입니다. 마크다운 파일이 든 폴더 하나이고, 통째로 git 저장소나 압축 파일로 주고받는 단위입니다. 번들 안에는 2절의 위키 층, 곧 정리된 문서 파일(개념 문서, 목차 `index.md`, 변경 기록 `log.md`)만 들어갑니다.

**원본 자료**는 번들 밖에 둡니다. 이 부록에서는 `data/raw/`입니다. 기존 위키 문서의 근거인 회의 원문 메모와 비서가 새로 받은 자료가 들어 있고, 위키에 넣은 뒤에도 원문은 그대로 둡니다. 개념 문서의 `sources`가 원본이 있는 곳을 가리킵니다(5-1).

**스키마**도 번들 밖에 둡니다. 이 부록에서는 코딩 에이전트용 규칙 파일 `data/AGENTS.md`(부록 02 1절), 비서의 시스템 프롬프트(부록 02), 스킬의 `SKILL.md`(부록 03)가 스키마입니다.

이 절에서 보는 폴더 구조, frontmatter, 목차와 변경 기록의 모양은 스키마 자체가 아니라, 규칙대로 쓴 결과가 번들에 남은 모습입니다. 그 가운데 모든 번들이 지키는 공통 부분(frontmatter의 `type`, 예약 파일 이름, 날짜 제목)을 정한 것이 OKF SPEC입니다.

번들 안의 `.md` 파일 하나가 **개념 문서(concept)** 하나이고, 번들 안 경로에서 `.md`를 뺀 것이 **개념 ID** 입니다. `index.md`(목차)와 `log.md`(변경 기록)는 예약된 이름이라 개념 문서로 쓰지 않습니다.

<img src="images/04_okf_bundle.png" width="1050" alt="번들 폴더 안의 개념 문서는 경로가 곧 개념 ID이고, frontmatter와 본문으로 이루어지며 본문 링크로 다른 문서와 이어집니다.">

예시 번들은 가상 게임 회사 별빛픽셀의 위키 두 개입니다. OKF는 폴더를 어떻게 나눌지 정하지 않으므로 두 번들은 폴더를 나누는 기준이 다릅니다. 같은 형식으로 쓰면 읽는 쪽 코드는 그대로입니다.

- **studio(역할별):** 사람, 프로젝트, 회의, 업무 절차, 결정. 회사 업무 비서가 읽습니다. `meetings/`의 회의 문서는 회의 원문이 아니라 그날의 참석자, 결정, 할 일을 정리한 문서입니다.
- **game_data(자원별):** 데이터셋, 테이블, 지표. OKF가 처음 겨냥한 데이터 카탈로그입니다. 데이터 자체는 넣지 않고 설명과 실제 위치(`resource`)만 둡니다.

이 부록의 `data/` 폴더입니다. 세 층이 폴더로 나뉘어 있고, 번들 두 개는 `bundles/` 아래에 있습니다. 오른쪽은 각 항목이 하는 일입니다.

~~~text
data/
├── AGENTS.md                          스키마: 코딩 에이전트가 읽는 위키 비서 규칙(부록 02)
├── raw/                               원본 자료: 읽기만 하고 고치지 않음. 위키에 넣은 뒤에도 그대로 둠
│   ├── meeting_2026-10-13_weekly.md   회의 원문 메모(회의 정리 문서와 결정 문서의 근거)
│   ├── meeting_2026-10-20_weekly.md   회의 원문 메모
│   ├── meeting_2026-10-27_weekly.md   회의 원문 메모
│   ├── 01_meeting_2026-10-31.md       새 자료: 긴급 회의 메모(부록 03에서 넣음)
│   ├── 02_mail_new_member.txt         새 자료: 새 팀원 소개 메일
│   ├── 03_data_memo_arppu.md          새 자료: 새 지표 정의 메모
│   ├── 04_data_memo_conversion.md     새 자료: 지표 중단 메모
│   ├── 05_ad_mail.txt                 새 자료: 광고 메일(넣지 않음)
│   ├── 06_lore_memo.md                새 자료: 세계관 메모(어느 번들에도 맞지 않아 새 번들로)
│   └── 07_equipment_memo.md           새 자료: 장비 목록 메모(업무 위키에 맞지만 폴더가 없어 새 폴더로)
└── bundles/                           위키: OKF 번들 두 개
    ├── studio/                        업무 위키(역할별 폴더)
    │   ├── index.md                   번들 맨 위 목차. okf_version을 적는 곳
    │   ├── log.md                     변경 기록(날짜 제목, 최신순)
    │   ├── people/                    사람: 팀원 한 명이 문서 하나
    │   │   ├── index.md               폴더 목차
    │   │   ├── yuna.md                윤하나, 대표
    │   │   ├── seoyeon.md             박서연, 서버 개발
    │   │   ├── minjun.md              이민준, 클라이언트 개발
    │   │   ├── doyun.md               강도윤, 게임 기획
    │   │   ├── hanbyeol.md            한별, 아트
    │   │   └── jiho.md                오지호, QA
    │   ├── projects/                  프로젝트: 목표, 일정, 담당
    │   │   ├── index.md               폴더 목차
    │   │   ├── chorong_launch.md      초롱 원정대 정식 출시
    │   │   └── chuseok_event.md       추석 이벤트(끝남, 기록용)
    │   ├── meetings/                  회의: 원문이 아니라 결정과 할 일을 정리한 문서
    │   │   ├── index.md               폴더 목차
    │   │   ├── 2026-10-13_weekly.md   결제 모듈 결정, 추석 이벤트 정리
    │   │   ├── 2026-10-20_weekly.md   출시 체크리스트 점검
    │   │   └── 2026-10-27_weekly.md   심사 제출일과 출시일 확정
    │   ├── playbooks/                 업무 절차: 순서, 연락처, 명령어
    │   │   ├── index.md               폴더 목차
    │   │   ├── launch_checklist.md    정식 출시 전 열두 항목
    │   │   ├── server_incident.md     서버 장애 대응(만료된 문서)
    │   │   └── vacation_request.md    휴가 신청과 인수인계
    │   └── decisions/                 결정: 무엇을 왜 정했는지
    │       ├── index.md               폴더 목차
    │       └── payment_module.md      스토어 기본 결제를 고른 이유
    └── game_data/                     데이터 위키(자원별 폴더). 데이터 자체는 넣지 않음
        ├── index.md                   번들 맨 위 목차
        ├── log.md                     변경 기록
        ├── datasets/                  데이터셋: 로그가 어디에 무엇이 있는지
        │   ├── index.md               폴더 목차
        │   └── game_logs.md           CBT 기간의 게임 로그
        ├── tables/                    테이블: 열 설명과 실제 위치(resource)
        │   ├── index.md               폴더 목차
        │   ├── players.md             플레이어 한 명이 한 행
        │   ├── sessions.md            접속 한 번이 한 행
        │   └── purchases.md           결제 한 번이 한 행
        └── metrics/                   지표: 정의, 계산 방법, 최근 값
            ├── index.md               폴더 목차
            ├── dau.md                 DAU
            ├── d7_retention.md        7일 잔존율
            └── paying_conversion.md   결제 전환율(초안)
~~~

### 4-2. 개념 문서에 넣는 것

개념 문서 하나는 **대상 하나(사람, 프로젝트, 테이블, 지표, 절차)에 대해 정리된 지식**입니다. 원본 자료를 그대로 옮긴 것이 아니라, 사람과 에이전트가 바로 쓸 수 있게 정리한 사실, 정의, 절차를 담습니다. `type`은 정해진 목록이 없고 만드는 쪽이 정합니다.

| type 예 | 본문에 넣는 것 | 이 부록의 예 |
|---|---|---|
| Person | 역할, 맡은 일, 관련 문서 링크 | `studio/people/seoyeon` |
| Project | 목표, 일정, 담당, 남은 일 | `studio/projects/chorong_launch` |
| Playbook | 순서대로 할 일, 연락처, 확인과 복구 명령어 | `studio/playbooks/server_incident` |
| Decision | 무엇을 정했고 왜 정했는지 | `studio/decisions/payment_module` |
| Table | 무엇이 들어 있는지, 열 설명(`# Schema`), 실제 위치(`resource`) | `game_data/tables/players` |
| Metric | 정의, 계산 방법(`# Computation`), 최근 값 | `game_data/metrics/d7_retention` |

넣지 않는 것도 있습니다.

- **원본 데이터 자체:** 표의 행이나 로그는 넣지 않고, `resource`로 실제 위치를 가리킵니다.
- **비밀 정보:** 비밀번호와 API 키는 넣지 않습니다. 번들은 여러 사람과 도구가 읽습니다.
- **원본 자료 통째:** 회의록이나 메일을 그대로 붙이지 않고, 정리한 사실만 적은 뒤 출처(`sources`)로 원본을 가리킵니다.

어떤 문서부터 만들지는 에이전트가 자주 받을 질문으로 정합니다. "누가 담당이지?"가 많으면 사람 문서, "어떻게 하지?"가 많으면 절차 문서부터 만듭니다. 절차 문서에는 명령어도 넣되, 그 명령을 쓰는 단계 안에 넣습니다.

업무 절차 문서입니다. 장애가 났을 때의 순서를 적고, 확인과 복구 명령어는 그 명령을 쓰는 단계(1번, 4번) 안에 넣었습니다.

`studio/playbooks/server_incident.md`

~~~markdown
---
type: Playbook
title: 서버 장애 대응
description: 서버 장애가 났을 때의 순서, 확인과 복구 명령어, 연락처.
tags: [server, incident]
status: stable
stale_after: "2026-10-01T00:00:00+09:00"
generated: { by: "human:seoyeon", at: "2026-07-01T10:00:00+09:00" }
---
# 순서

1. 아래 명령으로 게임 서버 상태를 확인합니다. 응답이 없거나 오류가 나오면 장애입니다.

   ```bash
   curl -s https://status.starpixel.example/api/health
   ```

2. 상태 페이지에 '점검 중'을 올립니다.
3. 1차 담당 [박서연](/people/seoyeon.md)에게 전화합니다.
4. 1차 담당이 아래 명령으로 게임 서버를 다시 시작합니다.

   ```bash
   ssh game-1 "sudo systemctl restart chorong-server"
   ```

5. 30분 안에 복구되지 않으면 [윤하나](/people/yuna.md)에게 알립니다.

# 연락처

1차 담당 전화번호는 사내 연락망 7번입니다. 클라우드 업체 지원 번호는 계약서 부록에 있습니다.
~~~

데이터 위키의 테이블 문서입니다. 데이터 자체가 아니라 열 설명과 실제 위치(`resource`)를 담습니다.

`game_data/tables/players.md`

~~~markdown
---
type: Table
title: players
description: 플레이어 한 명이 한 행.
resource: bigquery://starpixel/cbt_logs/players
tags: [cbt, players]
---
# Schema

| 열 | 형식 | 설명 |
|---|---|---|
| `player_id` | INTEGER | 플레이어 번호 |
| `joined_on` | TEXT | 가입한 날(YYYY-MM-DD) |
| `country` | TEXT | 국가 코드(KR, JP, TW) |
| `is_test` | INTEGER | 사내 테스트 계정이면 1 |

# Joins

[sessions](/tables/sessions.md), [purchases](/tables/purchases.md)와 `player_id`로 잇습니다.
~~~

### 4-3. frontmatter와 본문

개념 문서는 맨 위 `---` 두 줄 사이의 YAML frontmatter와 그 아래의 마크다운 본문으로 나뉩니다. 프로젝트 문서 하나를 봅니다.

`studio/projects/chorong_launch.md`

~~~markdown
---
type: Project
title: 초롱 원정대 출시
description: 모바일 게임 초롱 원정대의 정식 출시 프로젝트.
tags: [launch, mobile]
status: stable
sources:
  - id: weekly-1027
    resource: /meetings/2026-10-27_weekly.md
    title: 10월 27일 주간 회의
generated: { by: "starpixel-secretary/0.1", at: "2026-10-27T18:00:00+09:00" }
verified: { by: "human:yuna", at: "2026-10-28T09:30:00+09:00" }
---
# 개요

초롱이가 등불을 밝히며 밤하늘 섬을 탐험하는 모바일 퍼즐 어드벤처입니다. 담당은 [윤하나](/people/yuna.md)입니다.

# 일정

* 스토어 심사 제출: 11월 6일[^weekly-1027]
* 정식 출시: 11월 20일[^weekly-1027]

# 남은 일

출시 전 남은 항목은 [출시 체크리스트](/playbooks/launch_checklist.md)에서 관리합니다.

[^weekly-1027]: 10월 27일 주간 회의
~~~

본문의 `[^weekly-1027]`은 각주이고, frontmatter의 `sources`에서 같은 `id`를 가진 항목이 그 근거 자료입니다.

### 4-4. 목차 파일과 변경 기록

`index.md`는 폴더에 무엇이 있는지 한 줄씩 적은 목차입니다. 에이전트는 목차부터 읽고 필요한 문서만 엽니다(점진적 공개, progressive disclosure). 번들 맨 위 `index.md`만 frontmatter에 `okf_version`을 적을 수 있습니다. `log.md`는 `## YYYY-MM-DD` 날짜 제목 아래에 바뀐 내용을 최신순으로 적습니다.

둘 다 어느 폴더에나 둘 수 있는 선택 파일이지만, 이 부록은 목차는 폴더마다, 변경 기록은 번들 맨 위에 하나만 둡니다. 목차는 문서를 찾을 때마다 따라 내려가므로 폴더마다 있어야 편하고, 변경 기록은 최근에 무엇이 바뀌었는지 한곳에서 보는 것이 목적이기 때문입니다. 기록 한 줄마다 바뀐 문서 링크가 붙어 있어 어느 폴더의 변경인지도 보입니다. 위키가 아주 커서 변경 기록이 너무 길어지거나 폴더마다 담당 팀이 다르면 폴더마다 `log.md`를 따로 둡니다.

`studio/index.md`

~~~markdown
---
okf_version: "0.2"
---
# 별빛픽셀 업무 위키

* [사람](people/) - 팀원과 맡은 일
* [프로젝트](projects/) - 진행 중인 프로젝트와 일정
* [회의](meetings/) - 주간 회의록
* [업무 절차](playbooks/) - 출시, 장애, 휴가 절차
* [결정](decisions/) - 되돌리기 어려운 결정과 그 이유
~~~

`studio/log.md`

~~~markdown
# 업무 위키 변경 기록

## 2026-10-27
* **Update**: [초롱 원정대 출시](/projects/chorong_launch.md)에 10월 27일 회의 결과(스토어 심사 제출일)를 반영했습니다.
* **Creation**: [10월 27일 주간 회의](/meetings/2026-10-27_weekly.md)를 만들었습니다.

## 2026-10-13
* **Creation**: [결제 모듈 선택](/decisions/payment_module.md) 결정을 기록했습니다.

## 2026-09-01
* **Initialization**: 사람, 프로젝트, 회의, 업무 절차, 결정 폴더를 만들었습니다.
~~~

### 4-5. 문서 사이의 링크

문서끼리는 보통 마크다운 링크로 잇습니다. SPEC은 번들 맨 위를 기준으로 `/`부터 적는 링크를 권장합니다(문서를 옮겨도 덜 깨집니다). 링크에는 종류가 없고, 관계는 주변 문장이 말합니다.

## 5. 믿을 수 있는 지식

### 5-1. 출처, 검증, 만료

에이전트가 쓰고 고치는 위키는 "어디서 왔는지, 누가 확인했는지, 아직 맞는지"를 문서 안에 적어야 믿고 쓸 수 있습니다. OKF v0.2는 이를 위한 키를 정해 두었습니다. 모두 선택 키라 없어도 번들은 유효합니다.

<img src="images/05_okf_trust.png" width="1050" alt="출처(sources)와 각주로 근거를 잇고, 생성 기록과 검증 기록으로 신뢰 단계를 정하며, 상태와 stale_after로 수명을 표시합니다.">

- **출처(sources):** 문서의 근거 목록입니다. 항목의 `id`를 본문 각주 이름으로 씁니다. 근거는 번들 밖 원본 자료(`raw://...`)일 수도, 같은 번들의 다른 문서(`/meetings/...`)일 수도 있어서, 원문 메모, 회의 정리 문서, 프로젝트 문서 순으로 거슬러 올라갈 수 있습니다.
- **생성 기록(generated):** 누가(`by`) 언제(`at`) 썼는지입니다. 사람은 `human:아이디`, 자동 작업은 `process:아이디`, 에이전트는 `이름/버전`으로 적습니다.
- **검증 기록(verified):** 누가 언제 내용을 확인했는지입니다. 쓴 사람과 확인한 사람은 다를 수 있습니다. 예: 프로젝트 문서는 에이전트가 쓰고(`generated: starpixel-secretary/0.1`) 윤하나가 확인했습니다(`verified: { by: "human:yuna", at: "2026-10-28T09:30:00+09:00" }`).
- **신뢰 단계:** `verified`로 정합니다. 없으면 검증 전, `human:`이 확인했으면 사람 검토, 그 밖의 확인만 있으면 기계 확인입니다.
- **상태(status):** `draft`(초안), `stable`(기본값), `deprecated`(기록용)입니다.
- **만료 시점(stale_after):** 이 시각이 지나면 내용이 낡았을 수 있습니다. 예: 서버 장애 대응 문서는 `stale_after: "2026-10-01T00:00:00+09:00"`이라, 시나리오의 오늘(2026-11-02)에는 이미 지났습니다.

### 5-2. 예시 번들에서 보기

예시 번들의 문서 여섯 개를 위 기준으로 판정하면 다음과 같습니다. 신뢰 단계(`verified`)와 상태(`status`), 만료(`stale_after`)는 서로 따로 판정합니다. 그래서 사람이 검토한 문서도 만료될 수 있습니다.

| 문서 | frontmatter에서 볼 곳 | 판정 | 뜻 |
|---|---|---|---|
| `studio/projects/chorong_launch` | `verified`: human:yuna | 사람 검토 | 사람이 내용을 확인함 |
| `game_data/datasets/game_logs` | `verified`: process:log-export | 기계 확인 | 자동 작업만 확인함 |
| `game_data/metrics/dau` | `verified` 없음 | 검증 전 | 아무도 확인하지 않음 |
| `game_data/metrics/paying_conversion` | `status`: draft | 초안 | 아직 쓰는 중이라 빠진 내용이 있음 |
| `studio/projects/chuseok_event` | `status`: deprecated | 기록용 | 끝난 일이라 지금 상태로 쓰면 안 됨 |
| `studio/playbooks/server_incident` | `stale_after`: 2026-10-01 | 만료 | 이 날짜가 시나리오의 오늘(2026-11-02)보다 앞이라 내용이 낡았을 수 있음 |

부록 02의 비서는 이 판정을 답에 그대로 드러냅니다. 초안, 기록용, 만료 문서로 답할 때는 답의 첫 줄을 `주의:`로 시작하고, 마지막 출처 줄에는 문서마다 신뢰 단계를 적습니다.

## 6. 적합성 검사

**적합성**은 번들이 OKF 형식을 지켰는지입니다. 지키면 어떤 도구든 그 번들을 읽을 수 있습니다. SPEC 11절이 정한 조건은 세 가지뿐입니다.

1. **frontmatter 있음:** `index.md`, `log.md`를 뺀 모든 `.md` 파일 맨 위에 YAML 문법에 맞는 frontmatter가 있습니다.
2. **type 있음:** 그 frontmatter에 `type` 값이 있습니다.
3. **예약 파일 모양:** 폴더 목차 `index.md`에는 frontmatter를 넣지 않고(번들 맨 위 목차만 예외), `log.md`의 날짜 제목은 `## 2026-11-02`처럼 씁니다.

| 예 | 결과 |
|---|---|
| `type` 없이 `title`만 있는 문서 | 위반(조건 2) |
| 날짜 제목을 `## 11월 2일`로 쓴 `log.md` | 위반(조건 3) |
| 번들 폴더 안에 둔 `AGENTS.md`(frontmatter 없음) | 위반(조건 1). 그래서 규칙 파일은 번들 밖에 둡니다 |
| `verified`가 없는 문서, 처음 보는 `type`이나 키가 있는 문서 | 통과 |
| 링크가 끊긴 문서, 목차가 없는 폴더 | 통과 |

세 조건만 지키면, 다른 것이 빠졌거나 낯설어도 읽는 쪽은 번들을 거부하면 안 됩니다. 끊긴 링크나 만료처럼 고치면 좋은 것은 적합성과 따로, 부록 03의 점검 스킬이 찾아 알립니다.

## 7. 핵심 정리

- **OKF가 하는 일:** 흩어진 지식을 사람과 에이전트가 함께 쓰고 어떤 도구에서도 같은 방식으로 읽도록 번들로 정리합니다. 업무 비서, 데이터 카탈로그, 운영 절차서, 개인 지식 비서에 씁니다.
- **LLM 위키:** 원본 자료(읽기만), 위키(LLM이 쓰고 고침), 스키마(규칙)의 세 층과 넣기, 묻기, 점검의 세 동작입니다.
- **OKF:** YAML frontmatter가 붙은 마크다운 폴더입니다. 필수 키는 `type` 하나이고, 만드는 쪽과 읽는 쪽이 독립이며, 플랫폼이 아니라 형식입니다.
- **번들 구조:** 파일 경로가 개념 ID, `index.md`는 목차, `log.md`는 변경 기록, 문서 사이는 `/`부터 적는 링크로 잇습니다. 개념 문서에는 대상 하나에 대해 정리한 사실, 정의, 절차(명령어 포함)를 넣고, 원본 데이터와 비밀 정보는 넣지 않습니다.
- **믿을 수 있는 지식:** `sources`와 각주로 근거를, `generated`와 `verified`로 신뢰 단계를, `status`와 `stale_after`로 수명을 표시합니다. 신뢰 단계와 상태, 만료는 따로 판정합니다.
- **적합성:** 예약 파일이 아닌 `.md`마다 frontmatter와 `type`이 있고 `index.md`, `log.md`가 정해진 모양이면 유효한 번들입니다. 규칙 파일처럼 frontmatter가 없는 `.md`는 번들 밖에 둡니다.
- 부록 02는 신뢰 단계와 만료 기준을 코드로 옮긴 함수를 `okf_utils.py`에서 가져다 씁니다.